# Tech Challenge — Frente 2: Remuneração e valorização profissional

Este notebook lê as tabelas Parquet da camada Gold diretamente do Amazon S3,
prepara os dados e gera as análises de remuneração por senioridade, função,
Inteligência Artificial, região, modelo de trabalho, linguagem, nuvem e escolaridade.

As comparações são descritivas. Diferenças salariais não representam, por si só,
relações causais, pois também podem refletir senioridade, experiência, cargo,
localização e características das empresas.

In [ ]:
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 15

In [ ]:
# 1. Bibliotecas e inicialização da sessão
from pyspark.context import SparkContext
from awsglue.context import GlueContext

from io import BytesIO
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sc = SparkContext.getOrCreate()
glue_context = GlueContext(sc)
spark = glue_context.spark_session

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 10

print("Spark:", spark.version)

In [ ]:
# 2. Parâmetros e leitura dos Parquet no S3
S3_DIR = "s3://tech-challenge-014478672967/gold/frente_2_remuneracao"

CAMINHOS = {
    "fato": f"{S3_DIR}/ft_remuneracao_profissional_v1/",
    "evolucao": f"{S3_DIR}/agg_remuneracao_evolucao_anual_v1/",
    "segmentos": f"{S3_DIR}/agg_remuneracao_segmento_v1/",
}

ft = spark.read.parquet(CAMINHOS["fato"]).toPandas()
evo = spark.read.parquet(CAMINHOS["evolucao"]).toPandas()
seg = spark.read.parquet(CAMINHOS["segmentos"]).toPandas()

print(f"Fato .............. {ft.shape[0]:>6,} linhas x {ft.shape[1]} colunas")
print(f"Evolução anual .... {evo.shape[0]:>6,} linhas x {evo.shape[1]} colunas")
print(f"Segmentos ......... {seg.shape[0]:>6,} linhas x {seg.shape[1]} colunas")

display(ft.head(3))

In [ ]:
# 3. Validação das colunas utilizadas
COLUNAS_FT = {
    "ano_pesquisa",
    "tem_salario",
    "salario_medio",
    "atua_com_dados",
    "funcao_dados",
    "senioridade",
    "grupo_uso_ia",
    "regiao",
    "modelo_trabalho",
    "linguagem_principal",
    "cloud_preferida",
    "escolaridade",
}

COLUNAS_EVO = {
    "ano_pesquisa",
    "salario_mediano",
    "salario_primeiro_quartil",
    "salario_terceiro_quartil",
}

faltantes_ft = sorted(COLUNAS_FT.difference(ft.columns))
faltantes_evo = sorted(COLUNAS_EVO.difference(evo.columns))

if faltantes_ft:
    raise KeyError(
        "Colunas ausentes em ft_remuneracao_profissional_v1: "
        + ", ".join(faltantes_ft)
    )

if faltantes_evo:
    raise KeyError(
        "Colunas ausentes em agg_remuneracao_evolucao_anual_v1: "
        + ", ".join(faltantes_evo)
    )

print("Validação concluída: todas as colunas necessárias foram encontradas.")

In [ ]:
# 4. Cores e funções auxiliares
AZUL = "#2563EB"
VERDE = "#16A34A"
LARANJA = "#EA580C"
CINZA = "#9CA3AF"
ROXO = "#7C3AED"

PALETA = [
    "#2563EB",
    "#60A5FA",
    "#16A34A",
    "#7C3AED",
    "#EA580C",
    "#0F766E",
    "#9333EA",
]

SALVAR_GRAFICOS_S3 = True
BUCKET_GRAFICOS = "tech-challenge-014478672967"
PREFIXO_GRAFICOS = "artefatos/frente_2_remuneracao"

s3 = boto3.client("s3")


def brl(valor):
    """Formata valores monetários no padrão brasileiro."""
    if pd.isna(valor):
        return "N/D"

    return (
        f"R$ {valor:,.0f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def inteiro_br(valor):
    """Formata números inteiros com ponto como separador de milhares."""
    return f"{int(valor):,}".replace(",", ".")


def limpa_eixo(ax, grade="y"):
    """Remove elementos visuais desnecessários e adiciona grade leve."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.grid(axis=grade, alpha=0.20)


def para_booleano(serie):
    """Converte booleanos, números 0/1 e textos para True/False."""
    valores = serie.astype("string").str.strip().str.lower()

    return valores.isin([
        "1",
        "1.0",
        "true",
        "t",
        "sim",
        "yes",
    ])


def media_salario_por(dataframe, coluna, ordem):
    """Calcula o salário médio seguindo uma ordem de categorias."""
    return (
        dataframe.loc[dataframe[coluna].isin(ordem)]
        .groupby(coluna, observed=True)["salario_medio"]
        .mean()
        .reindex(ordem)
    )


def rotula_barras_horizontais(ax, valores):
    """Adiciona rótulos monetários em barras horizontais."""
    valores_validos = pd.Series(valores).dropna()

    if valores_validos.empty:
        return

    deslocamento = valores_validos.max() * 0.015

    for barra, valor in zip(ax.patches, valores):
        if pd.notna(valor):
            ax.text(
                valor + deslocamento,
                barra.get_y() + barra.get_height() / 2,
                brl(valor),
                va="center",
                fontsize=9,
                fontweight="bold",
            )


def concluir_grafico(fig, nome_arquivo):
    """Ajusta, salva o gráfico no S3, exibe e fecha a figura."""
    fig.tight_layout()

    if SALVAR_GRAFICOS_S3:
        buffer = BytesIO()
        fig.savefig(
            buffer,
            format="png",
            dpi=180,
            bbox_inches="tight",
            facecolor="white",
        )
        buffer.seek(0)

        chave = f"{PREFIXO_GRAFICOS}/{nome_arquivo}.png"

        try:
            s3.put_object(
                Bucket=BUCKET_GRAFICOS,
                Key=chave,
                Body=buffer.getvalue(),
                ContentType="image/png",
            )
            print(f"Gráfico salvo em s3://{BUCKET_GRAFICOS}/{chave}")
        except Exception as erro:
            print(
                "O gráfico foi exibido, mas não pôde ser salvo no S3: "
                f"{erro}"
            )
        finally:
            buffer.close()

    plt.show()
    plt.close(fig)

In [ ]:
# 5. Preparação e padronização dos dados
ft = ft.copy()
evo = evo.copy()
seg = seg.copy()

ft["ano_pesquisa"] = pd.to_numeric(
    ft["ano_pesquisa"], errors="coerce"
).astype("Int64")

ft["salario_medio"] = pd.to_numeric(
    ft["salario_medio"], errors="coerce"
)

ft["tem_salario_bool"] = para_booleano(ft["tem_salario"])
ft["atua_com_dados_bool"] = para_booleano(ft["atua_com_dados"])

colunas_numericas_evo = [
    "ano_pesquisa",
    "salario_medio",
    "salario_mediano",
    "salario_primeiro_quartil",
    "salario_terceiro_quartil",
]

for coluna in colunas_numericas_evo:
    if coluna in evo.columns:
        evo[coluna] = pd.to_numeric(evo[coluna], errors="coerce")

sal = ft.loc[
    ft["tem_salario_bool"]
    & ft["salario_medio"].notna()
    & ft["salario_medio"].gt(0)
].copy()

print(f"Fato .............. {ft.shape[0]:>6,} linhas x {ft.shape[1]} colunas")
print(
    f"  com salário ..... {sal.shape[0]:>6,} linhas "
    f"({sal.shape[0] / ft.shape[0]:.0%})"
)
print(f"Evolução anual .... {evo.shape[0]:>6,} linhas")
print(f"Segmentos ......... {seg.shape[0]:>6,} linhas")

## 1. Perfil dos profissionais

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# Respondentes por ano
por_ano = (
    ft["ano_pesquisa"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

barras_ano = axes[0].bar(
    por_ano.index.astype(str),
    por_ano.values,
    color=AZUL,
    width=0.6,
)

axes[0].set_title("Respondentes por ano")
axes[0].set_xlabel("Ano da pesquisa")
axes[0].set_ylabel("Quantidade de respondentes")
axes[0].bar_label(
    barras_ano,
    labels=[inteiro_br(v) for v in por_ano.values],
    padding=4,
    fontsize=9,
)
axes[0].set_ylim(0, por_ano.max() * 1.15)

# Atuação direta com dados
atua = (
    ft["atua_com_dados_bool"]
    .map({True: "Atua com dados", False: "Não atua"})
    .value_counts()
    .reindex(["Atua com dados", "Não atua"], fill_value=0)
)
atua = atua[atua.gt(0)]

axes[1].pie(
    atua.values,
    labels=atua.index,
    autopct="%1.0f%%",
    colors=[VERDE, CINZA][:len(atua)],
    startangle=90,
    wedgeprops={"width": 0.45, "edgecolor": "white"},
)
axes[1].set_title("Atuação direta com dados")
axes[1].set_aspect("equal")

# Profissionais por função
func = ft["funcao_dados"].dropna().astype(str).str.strip()
func = func[func.ne("")].value_counts()

barras_funcao = axes[2].barh(
    func.index[::-1],
    func.values[::-1],
    color=LARANJA,
)
axes[2].set_title("Profissionais por função de dados")
axes[2].set_xlabel("Quantidade de profissionais")
axes[2].bar_label(
    barras_funcao,
    labels=[inteiro_br(v) for v in func.values[::-1]],
    padding=4,
    fontsize=9,
)
axes[2].set_xlim(0, func.max() * 1.20)

limpa_eixo(axes[0])
limpa_eixo(axes[2], grade="x")

concluir_grafico(fig, "01_perfil_profissionais")

## 2. Salário médio por senioridade

In [ ]:
ordem_senioridade = ["Júnior", "Pleno", "Sênior"]

salario_senioridade = (
    sal.loc[sal["senioridade"].isin(ordem_senioridade)]
    .groupby("senioridade", observed=True)["salario_medio"]
    .mean()
    .reindex(ordem_senioridade)
)

salario_plot = salario_senioridade.dropna()

if not salario_plot.empty:
    fig, ax = plt.subplots(figsize=(8, 4.6))

    cores_senioridade = {
        "Júnior": ROXO,
        "Pleno": AZUL,
        "Sênior": VERDE,
    }

    barras = ax.bar(
        salario_plot.index,
        salario_plot.values,
        color=[cores_senioridade[nivel] for nivel in salario_plot.index],
        width=0.6,
    )

    ax.set_title("Salário médio mensal por senioridade")
    ax.set_xlabel("Senioridade")
    ax.set_ylabel("Salário médio mensal")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
    )
    ax.bar_label(
        barras,
        labels=[brl(valor) for valor in salario_plot.values],
        padding=5,
        fontsize=10,
        fontweight="bold",
    )
    ax.set_ylim(0, salario_plot.max() * 1.18)

    limpa_eixo(ax)
    concluir_grafico(fig, "02_salario_senioridade")

junior = salario_senioridade.get("Júnior")
senior = salario_senioridade.get("Sênior")

if pd.notna(junior) and pd.notna(senior) and junior > 0:
    multiplicador = senior / junior
    print(
        f"Um profissional Sênior ganha, em média, "
        f"{multiplicador:.1f} vez(es) o salário de um Júnior."
    )
else:
    print("Não existem dados suficientes para comparar Júnior e Sênior.")

## 3. Evolução salarial entre os anos

In [ ]:
e = (
    evo.dropna(
        subset=[
            "ano_pesquisa",
            "salario_mediano",
            "salario_primeiro_quartil",
            "salario_terceiro_quartil",
        ]
    )
    .sort_values("ano_pesquisa")
    .copy()
)

anos = e["ano_pesquisa"].astype(int).to_numpy()
mediana = e["salario_mediano"].astype(float).to_numpy()
primeiro_quartil = e["salario_primeiro_quartil"].astype(float).to_numpy()
terceiro_quartil = e["salario_terceiro_quartil"].astype(float).to_numpy()

if len(e) > 0:
    fig, ax = plt.subplots(figsize=(9, 4.8))

    ax.fill_between(
        anos,
        primeiro_quartil,
        terceiro_quartil,
        color=AZUL,
        alpha=0.15,
        label="Faixa Q1–Q3 (50% do mercado)",
    )
    ax.plot(anos, mediana, marker="o", color=AZUL, linewidth=2.5, label="Mediana")
    ax.plot(
        anos,
        terceiro_quartil,
        marker="^",
        color=VERDE,
        linewidth=1.8,
        linestyle="--",
        label="3º quartil (Q3)",
    )
    ax.plot(
        anos,
        primeiro_quartil,
        marker="v",
        color=LARANJA,
        linewidth=1.8,
        linestyle="--",
        label="1º quartil (Q1)",
    )

    espaco_rotulo = mediana.max() * 0.035
    for ano, valor in zip(anos, mediana):
        ax.text(
            ano,
            valor + espaco_rotulo,
            brl(valor),
            ha="center",
            fontsize=9,
            fontweight="bold",
            color=AZUL,
        )

    ax.set_title("Evolução dos salários — quartis por ano de pesquisa")
    ax.set_xlabel("Ano da pesquisa")
    ax.set_ylabel("Salário mensal")
    ax.set_xticks(anos)
    ax.set_xticklabels([str(ano) for ano in anos])
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
    )
    ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
    ax.set_ylim(0, terceiro_quartil.max() * 1.15)

    limpa_eixo(ax)
    concluir_grafico(fig, "03_evolucao_salarios")

mediana_por_ano = e.set_index("ano_pesquisa")["salario_mediano"]

print("Mediana salarial por ano:")
for ano, valor in mediana_por_ano.items():
    print(f"  {int(ano)}: {brl(valor)}")

if (
    2023 in mediana_por_ano.index
    and 2025 in mediana_por_ano.index
    and mediana_por_ano.loc[2023] > 0
):
    variacao = mediana_por_ano.loc[2025] / mediana_por_ano.loc[2023] - 1
    print(f"Variação da mediana entre 2023 e 2025: {variacao:+.1%}")
else:
    print("Não existem dados suficientes para calcular a variação 2023–2025.")

## 4. Salário médio por função de dados

In [ ]:
ordem_f = [
    "Ciência de Dados/ML/AI",
    "Engenharia de Dados",
    "Análise de Dados/BI",
    "Outras frentes de dados",
]

g = media_salario_por(sal, "funcao_dados", ordem_f).dropna().sort_values()

if not g.empty:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    cores = plt.cm.Blues(np.linspace(0.45, 0.85, len(g)))

    ax.barh(g.index, g.values, color=cores)
    rotula_barras_horizontais(ax, g.values)

    ax.set_title("Salário médio por função de dados")
    ax.set_xlabel("Salário médio mensal")
    ax.set_xlim(0, g.max() * 1.20)
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
    )

    limpa_eixo(ax, grade="x")
    concluir_grafico(fig, "04_salario_funcao_dados")
else:
    print("Não existem dados salariais por função.")

## 5. Adoção de Inteligência Artificial e salário

In [ ]:
def padroniza_grupo_ia(serie):
    return (
        serie.astype("string")
        .str.strip()
        .replace({
            "Nao usa IA": "Não usa IA",
            "Não Usa IA": "Não usa IA",
            "Usa ia": "Usa IA",
        })
    )


grupo_ia_ft = padroniza_grupo_ia(ft["grupo_uso_ia"])

adocao_ia = (
    grupo_ia_ft[grupo_ia_ft.isin(["Usa IA", "Não usa IA"])]
    .value_counts()
    .reindex(["Usa IA", "Não usa IA"], fill_value=0)
)
adocao_ia = adocao_ia[adocao_ia.gt(0)]

sal_ia = sal.copy()
sal_ia["grupo_ia_padrao"] = padroniza_grupo_ia(sal_ia["grupo_uso_ia"])

gi = (
    sal_ia.loc[sal_ia["grupo_ia_padrao"].isin(["Usa IA", "Não usa IA"])]
    .groupby("grupo_ia_padrao", observed=True)["salario_medio"]
    .mean()
    .reindex(["Não usa IA", "Usa IA"])
    .dropna()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

axes[0].pie(
    adocao_ia.values,
    labels=adocao_ia.index,
    autopct="%1.0f%%",
    colors=[VERDE if categoria == "Usa IA" else CINZA for categoria in adocao_ia.index],
    startangle=90,
    wedgeprops={"width": 0.45, "edgecolor": "white"},
)
axes[0].set_title("Adoção de IA entre os respondentes")
axes[0].set_aspect("equal")

cores_ia = [CINZA if categoria == "Não usa IA" else VERDE for categoria in gi.index]
barras_ia = axes[1].bar(gi.index, gi.values, color=cores_ia, width=0.55)
axes[1].bar_label(
    barras_ia,
    labels=[brl(valor) for valor in gi.values],
    padding=5,
    fontsize=10,
    fontweight="bold",
)
axes[1].set_title("Salário médio: uso de IA")
axes[1].set_ylabel("Salário médio mensal")
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
)
if not gi.empty:
    axes[1].set_ylim(0, gi.max() * 1.18)

limpa_eixo(axes[1])
concluir_grafico(fig, "05_adocao_ia_salario")

if "Usa IA" in gi.index and "Não usa IA" in gi.index and gi["Não usa IA"] > 0:
    premio_ia = gi["Usa IA"] / gi["Não usa IA"] - 1
    print(f"Diferença salarial de quem usa IA: {premio_ia:+.0%}.")
else:
    print("Não existem dados suficientes para comparar os grupos de IA.")

## 6. Volume e salário por região

In [ ]:
ordem_r = ["Sudeste", "Sul", "Nordeste", "Centro-oeste", "Norte"]

ft_regiao = ft.copy()
sal_regiao = sal.copy()

mapa_regiao = {
    "Centro-Oeste": "Centro-oeste",
    "Centro Oeste": "Centro-oeste",
}

ft_regiao["regiao_padrao"] = ft_regiao["regiao"].replace(mapa_regiao)
sal_regiao["regiao_padrao"] = sal_regiao["regiao"].replace(mapa_regiao)

vol = (
    ft_regiao.loc[ft_regiao["regiao_padrao"].isin(ordem_r), "regiao_padrao"]
    .value_counts()
    .reindex(ordem_r, fill_value=0)
)

salr = (
    sal_regiao.loc[sal_regiao["regiao_padrao"].isin(ordem_r)]
    .groupby("regiao_padrao", observed=True)["salario_medio"]
    .mean()
    .reindex(ordem_r)
    .dropna()
)

total_regioes = vol.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

barras_volume = axes[0].bar(vol.index, vol.values, color=AZUL, width=0.65)
axes[0].bar_label(
    barras_volume,
    labels=[f"{valor / total_regioes:.0%}" if total_regioes > 0 else "" for valor in vol.values],
    padding=4,
    fontsize=9,
)
axes[0].set_title("Volume de profissionais por região")
axes[0].set_ylabel("Quantidade de profissionais")
axes[0].tick_params(axis="x", rotation=20)
if vol.max() > 0:
    axes[0].set_ylim(0, vol.max() * 1.18)

barras_salario = axes[1].bar(salr.index, salr.values, color=VERDE, width=0.65)
axes[1].bar_label(
    barras_salario,
    labels=[brl(valor) for valor in salr.values],
    padding=5,
    fontsize=9,
    fontweight="bold",
)
axes[1].set_title("Salário médio por região")
axes[1].set_ylabel("Salário médio mensal")
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
)
axes[1].tick_params(axis="x", rotation=20)
if not salr.empty:
    axes[1].set_ylim(0, salr.max() * 1.18)

for ax in axes:
    limpa_eixo(ax)

concluir_grafico(fig, "06_regiao_volume_salario")

if total_regioes > 0:
    print(
        f"O Sudeste concentra {vol.get('Sudeste', 0) / total_regioes:.0%} "
        f"dos profissionais."
    )

## 7. Salário por modelo de trabalho

In [ ]:
ordem_m = ["Remoto", "Híbrido flexível", "Híbrido dias fixos", "Presencial"]

vol = (
    ft.loc[ft["modelo_trabalho"].isin(ordem_m), "modelo_trabalho"]
    .value_counts()
    .reindex(ordem_m, fill_value=0)
)
salm = media_salario_por(sal, "modelo_trabalho", ordem_m)

dados_modelo = pd.DataFrame({"volume": vol, "salario": salm}).dropna(
    subset=["salario"]
)
total_modelos = vol.sum()

if not dados_modelo.empty:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    x = np.arange(len(dados_modelo))

    cores_modelo = {
        "Remoto": VERDE,
        "Híbrido flexível": AZUL,
        "Híbrido dias fixos": ROXO,
        "Presencial": LARANJA,
    }

    barras = ax.bar(
        x,
        dados_modelo["salario"].values,
        width=0.60,
        color=[cores_modelo[categoria] for categoria in dados_modelo.index],
    )

    for barra, salario, quantidade in zip(
        barras,
        dados_modelo["salario"],
        dados_modelo["volume"],
    ):
        ax.text(
            barra.get_x() + barra.get_width() / 2,
            salario + dados_modelo["salario"].max() * 0.025,
            brl(salario),
            ha="center",
            fontsize=9,
            fontweight="bold",
        )

        percentual = quantidade / total_modelos if total_modelos > 0 else 0
        ax.text(
            barra.get_x() + barra.get_width() / 2,
            salario / 2,
            f"{percentual:.0%}\nda base",
            ha="center",
            va="center",
            fontsize=9,
            color="white",
            fontweight="bold",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(dados_modelo.index, rotation=12)
    ax.set_title("Salário médio por modelo de trabalho")
    ax.set_ylabel("Salário médio mensal")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
    )
    ax.set_ylim(0, dados_modelo["salario"].max() * 1.18)

    limpa_eixo(ax)
    concluir_grafico(fig, "07_salario_modelo_trabalho")
else:
    print("Não existem dados suficientes por modelo de trabalho.")

## 8. Salário por linguagem principal

In [ ]:
principais = [
    "SQL",
    "Python",
    "R",
    "Scala",
    "JavaScript",
    "SAS/Stata",
    "Visual Basic/VBA",
]

base_linguagem = sal.loc[sal["linguagem_principal"].isin(principais)].copy()
vol = base_linguagem["linguagem_principal"].value_counts().reindex(
    principais, fill_value=0
)
sall = (
    base_linguagem.groupby("linguagem_principal", observed=True)["salario_medio"]
    .mean()
    .reindex(principais)
)

dados_linguagem = (
    pd.DataFrame({"volume": vol, "salario": sall})
    .dropna(subset=["salario"])
    .query("volume > 0")
    .sort_values("salario")
)

if not dados_linguagem.empty:
    volume_min = dados_linguagem["volume"].min()
    volume_max = dados_linguagem["volume"].max()

    if volume_max == volume_min:
        normalizacao = np.full(len(dados_linguagem), 0.5)
    else:
        normalizacao = (
            dados_linguagem["volume"] - volume_min
        ) / (volume_max - volume_min)

    cores = plt.cm.Blues(0.35 + 0.60 * np.asarray(normalizacao))

    fig, ax = plt.subplots(figsize=(10, 4.8))
    barras = ax.barh(
        dados_linguagem.index,
        dados_linguagem["salario"],
        color=cores,
    )

    deslocamento = dados_linguagem["salario"].max() * 0.015
    for barra, salario, quantidade in zip(
        barras,
        dados_linguagem["salario"],
        dados_linguagem["volume"],
    ):
        ax.text(
            salario + deslocamento,
            barra.get_y() + barra.get_height() / 2,
            f"{brl(salario)} · {inteiro_br(quantidade)} prof.",
            va="center",
            fontsize=9,
        )

    ax.set_title("Salário médio por linguagem principal")
    ax.set_xlabel("Salário médio mensal")
    ax.set_xlim(0, dados_linguagem["salario"].max() * 1.32)
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
    )

    limpa_eixo(ax, grade="x")
    concluir_grafico(fig, "08_salario_linguagem_principal")
else:
    print("Não existem dados suficientes por linguagem.")

## 9. Preferência de nuvem

In [ ]:
clouds = [
    "Amazon Web Services (AWS)",
    "Google Cloud (GCP)",
    "Azure (Microsoft)",
    "Sem preferência / Não sei opinar",
]

rotulos_cloud = {
    "Amazon Web Services (AWS)": "AWS",
    "Google Cloud (GCP)": "GCP",
    "Azure (Microsoft)": "Azure",
    "Sem preferência / Não sei opinar": "Sem preferência",
}

base_cloud = ft.loc[ft["cloud_preferida"].isin(clouds)].copy()
vol = (
    base_cloud["cloud_preferida"]
    .value_counts()
    .reindex(clouds, fill_value=0)
    .rename(index=rotulos_cloud)
)

sc_cloud = (
    sal.loc[sal["cloud_preferida"].isin(clouds[:3])]
    .groupby("cloud_preferida", observed=True)["salario_medio"]
    .mean()
    .reindex(clouds[:3])
    .rename(index=rotulos_cloud)
    .dropna()
)

total_cloud = vol.sum()
mapa_cores_cloud = {
    "AWS": LARANJA,
    "GCP": AZUL,
    "Azure": ROXO,
    "Sem preferência": CINZA,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

barras_cloud = axes[0].bar(
    vol.index,
    vol.values,
    color=[mapa_cores_cloud[categoria] for categoria in vol.index],
    width=0.65,
)
axes[0].bar_label(
    barras_cloud,
    labels=[
        f"{inteiro_br(valor)}\n({valor / total_cloud:.0%})"
        if total_cloud > 0
        else ""
        for valor in vol.values
    ],
    padding=4,
    fontsize=9,
)
axes[0].set_title("Preferência de nuvem")
axes[0].set_ylabel("Quantidade de profissionais")
axes[0].tick_params(axis="x", rotation=12)
if vol.max() > 0:
    axes[0].set_ylim(0, vol.max() * 1.22)

barras_salario_cloud = axes[1].bar(
    sc_cloud.index,
    sc_cloud.values,
    color=[mapa_cores_cloud[categoria] for categoria in sc_cloud.index],
    width=0.55,
)
axes[1].bar_label(
    barras_salario_cloud,
    labels=[brl(valor) for valor in sc_cloud.values],
    padding=5,
    fontsize=9,
    fontweight="bold",
)
axes[1].set_title("Salário médio por nuvem preferida")
axes[1].set_ylabel("Salário médio mensal")
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
)
if not sc_cloud.empty:
    axes[1].set_ylim(0, sc_cloud.max() * 1.18)

for ax in axes:
    limpa_eixo(ax)

concluir_grafico(fig, "09_cloud_preferencia_salario")

quantidade_aws = vol.get("AWS", 0)
if total_cloud > 0:
    print(
        f"A AWS é a nuvem preferida por {inteiro_br(quantidade_aws)} profissionais "
        f"({quantidade_aws / total_cloud:.0%} entre os que responderam)."
    )

## 10. Salário médio por escolaridade

In [ ]:
ordem_e = [
    "Estudante de Graduação",
    "Graduação/Bacharelado",
    "Pós-graduação",
    "Mestrado",
    "Doutorado ou Phd",
]

sal_escolaridade = sal.copy()
sal_escolaridade["escolaridade_padrao"] = sal_escolaridade["escolaridade"].replace(
    {
        "Doutorado ou PhD": "Doutorado ou Phd",
        "Doutorado/PhD": "Doutorado ou Phd",
    }
)

g = (
    sal_escolaridade.loc[
        sal_escolaridade["escolaridade_padrao"].isin(ordem_e)
    ]
    .groupby("escolaridade_padrao", observed=True)["salario_medio"]
    .mean()
    .reindex(ordem_e)
    .dropna()
)

if not g.empty:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    cores = plt.cm.Greens(np.linspace(0.40, 0.85, len(g)))
    x = np.arange(len(g))

    barras = ax.bar(x, g.values, color=cores, width=0.62)
    ax.bar_label(
        barras,
        labels=[brl(valor) for valor in g.values],
        padding=5,
        fontsize=9,
        fontweight="bold",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(g.index, rotation=15, ha="right", fontsize=9)
    ax.set_title("Salário médio por escolaridade")
    ax.set_ylabel("Salário médio mensal")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda valor, _: f"R$ {valor / 1000:.0f} mil")
    )
    ax.set_ylim(0, g.max() * 1.18)

    limpa_eixo(ax)
    concluir_grafico(fig, "10_salario_escolaridade")
else:
    print("Não existem dados suficientes por escolaridade.")

## Observação metodológica

As análises apresentadas são descritivas. Por exemplo, uma diferença salarial
entre profissionais que usam e não usam IA não comprova que o uso de IA seja a
causa da diferença. O resultado pode ser influenciado por senioridade, cargo,
experiência, região, formação e outras características profissionais.